### 타이타닉 탑승자 데이터
| 컬럼명              | 설명                                                        |
| ---------------- | --------------------------------------------------------- |
| **survived**     | 생존 여부 (0 = 사망, 1 = 생존)                                    |
| **pclass**       | 선실 등급 (1 = 1등석, 2 = 2등석, 3 = 3등석)                         |
| **sex**          | 성별 ('male', 'female')                                     |
| **age**          | 나이                                                        |
| **sibsp**        | 함께 탑승한 형제/배우자 수 (siblings/spouses)                        |
| **parch**        | 함께 탑승한 부모/자녀 수 (parents/children)                         |
| **fare**         | 승객 요금                                                     |
| **embarked**     | 탑승 항구 코드 (C = Cherbourg, Q = Queenstown, S = Southampton) |
| **class**        | `pclass`와 동일하지만 문자열로 표시 ('First', 'Second', 'Third')      |
| **who**          | 승객 구분 ('man', 'woman', 'child')                           |
| **adult\_male**  | 성인 남성 여부 (True/False)                                     |
| **deck**         | 객실이 위치한 갑판 (A\~G 등, 일부 결측값 존재)                            |
| **embark\_town** | 탑승 도시명 (예: 'Cherbourg', 'Southampton')                    |
| **alive**        | 생존 여부 문자열 ('yes' 또는 'no')                                 |
| **alone**        | 혼자 탑승 여부 (True/False)                                     |

※ survived와 alive은 같은 컬럼이므로 alive제외하고 훈련해야함


In [104]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report

In [105]:
titanic = sns.load_dataset('titanic')
print(titanic.columns)
print(titanic.info())
print(titanic.head(2))

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null  

In [106]:
titanic.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [107]:
titanic.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [108]:
titanic.fillna({'age': titanic['age'].mean(), 'embarked': titanic['embarked'].mode()[0]}, inplace=True)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.000000,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.000000,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.000000,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.000000,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.000000,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.000000,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.000000,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,29.699118,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.000000,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [109]:
titanic['age'].isnull().sum()

np.int64(0)

In [110]:
titanic['embarked'].value_counts()

embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [111]:
titanic['embarked'].isnull().sum()

np.int64(0)

In [112]:
# titanic['deck'] = titanic['deck'].cat.add_categories('N')
titanic.fillna({'deck': 'N'}, inplace=True)
titanic['deck'].isnull().sum()

TypeError: Cannot setitem on a Categorical with a new category (N), set the categories first

In [ ]:
titanic['deck'].value_counts()

deck
N    688
C     59
B     47
D     33
E     32
A     15
F     13
G      4
Name: count, dtype: int64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(titanic.loc[:, ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'deck']], titanic['survived'], test_size=0.2, random_state=42)

In [ ]:
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          891 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     891 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         891 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [ ]:
titanic['sex'].value_counts()

sex
male      577
female    314
Name: count, dtype: int64

## LabelEncoder() 로 인코딩

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()

titanic_df1 = titanic.copy()

# 변환할 컬럼 목록
label_cols = ['sex', 'embarked', 'class', 'who','deck','embark_town']

for col in label_cols:
    titanic_df1[col] = le.fit_transform(titanic_df1[col])

In [ ]:
titanic_df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    int64  
 3   age          891 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     891 non-null    int64  
 8   class        891 non-null    int64  
 9   who          891 non-null    int64  
 10  adult_male   891 non-null    bool   
 11  deck         891 non-null    int64  
 12  embark_town  891 non-null    int64  
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(10), str(1)
memory usage: 92.4 KB


In [ ]:
titanic_df1[label_cols].head()

,sex,embarked,class,who,deck,embark_town
0,1,2,2,1,7,2
1,0,0,0,2,2,0
2,0,2,2,2,7,2
3,0,2,0,2,2,2
4,1,2,2,1,7,2


In [ ]:
titanic_df1['sex'].value_counts()

sex
1    577
0    314
Name: count, dtype: int64

In [ ]:
titanic_df1['deck'].value_counts()

deck
7    688
2     59
1     47
3     33
4     32
0     15
5     13
6      4
Name: count, dtype: int64

### OnHotEncoder(sparse_output=False, handle_unknown='ignore')

남 1 0
여 0 1

0 0 1 0 0 0 0 0
0 0 0 0 0 1 0 0

OneHotEncoder(sparse_output=False, handle_unknown='ignore')

| 파라미터                      | 설명                                                                            |
| ------------------------- | ----------------------------------------------------------------------------- |
| `sparse_output=False`     | 반환값을 **희소 행렬(sparse matrix)** 대신 **NumPy 배열**로 반환. <br> → DataFrame으로 변환하기 쉬움 |
| `handle_unknown='ignore'` | 학습 시 보지 못한 **새로운 범주가 등장해도 에러 없이 무시**. <br> → 안정적인 예측 및 인코딩 수행                 |


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

titanic_df2 = titanic.copy()

# 변환할 컬럼 목록
ohe_cols = ['sex', 'embarked', 'class', 'who','deck','embark_town']

ohe_result = pd.DataFrame(index=titanic_df2.index)
print(ohe_result.head())
print('ohe_result_len:', len(ohe_result))

for col in ohe_cols:
    encoded = ohe.fit_transform(titanic_df2[[col]])
    # print(encoded)

    col_names = ohe.get_feature_names_out([col])
    # print(col_names)

    encoded_df = pd.DataFrame(encoded, columns=col_names, index=titanic_df2.index)
    ohe_result = pd.concat([ohe_result, encoded_df], axis=1)
    # print(ohe_result.head())
    # break;

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]
ohe_result_len: 891


In [ ]:
print(len(ohe_result.columns))
ohe_result.head()

23


,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,class_First,class_Second,class_Third,who_child,who_man,...,deck_C,deck_D,deck_E,deck_F,deck_G,deck_N,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton,embark_town_nan
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [ ]:
# survived 와 alive 컬럼 2개가 생존여부를 의미(동일내용). 그래서 alive 컬럼 제거
titanic_df2.drop(columns=['alive'], inplace=True)


In [ ]:
titanic_df2 = pd.concat([titanic_df2, ohe_result], axis=1)

titanic_df2

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,deck_C,deck_D,deck_E,deck_F,deck_G,deck_N,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton,embark_town_nan
0,0,3,male,22.000000,1,0,7.2500,S,Third,man,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1,1,female,38.000000,1,0,71.2833,C,First,woman,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,1,3,female,26.000000,0,0,7.9250,S,Third,woman,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1,1,female,35.000000,1,0,53.1000,S,First,woman,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0,3,male,35.000000,0,0,8.0500,S,Third,man,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.000000,0,0,13.0000,S,Second,man,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
887,1,1,female,19.000000,0,0,30.0000,S,First,woman,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
888,0,3,female,29.699118,1,2,23.4500,S,Third,woman,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
889,1,1,male,26.000000,0,0,30.0000,C,First,man,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
print(titanic_df2.columns)
titanic_df2.head(1)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alone', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q',
       'embarked_S', 'class_First', 'class_Second', 'class_Third', 'who_child',
       'who_man', 'who_woman', 'deck_A', 'deck_B', 'deck_C', 'deck_D',
       'deck_E', 'deck_F', 'deck_G', 'deck_N', 'embark_town_Cherbourg',
       'embark_town_Queenstown', 'embark_town_Southampton', 'embark_town_nan'],
      dtype='str')


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,deck_C,deck_D,deck_E,deck_F,deck_G,deck_N,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton,embark_town_nan
0,0,3,male,22.0,1,0,7.25,S,Third,man,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


| 컬럼명           | 타입       | 인코딩 방식        | 설명                      |
| ------------- | -------- | ------------- | ----------------------- |
| `sex`         | str   | OneHotEncoder | 남/여 → 순서 의미 없음          |
| `embarked`    | str   | OneHotEncoder | 탑승 항구 (순서 없음)           |
| `class`       | category | LabelEncoder  | 1등, 2등, 3등 (순서 있음)      |
| `who`         | str   | OneHotEncoder | man/woman/child (순서 없음) |
| `deck`        | category | OneHotEncoder | 갑판 A\~G,N (순서 없음)       |
| `embark_town` | str   | OneHotEncoder | 도시명 (순서 없음)             |
| `alive`       | str   | LabelEncoder  | yes/no → 이진             |
| `adult_male`  | bool     | LabelEncoder  | True/False → 1/0        |
| `alone`       | bool     | LabelEncoder  | True/False → 1/0        |


In [114]:
titanic_df3 = titanic.copy()

titanic_df3 = titanic.copy()

# 1. LabelEncoder 적용할 컬럼
label_cols = ['class', 'adult_male', 'alone']
le = LabelEncoder()

for col in label_cols:
    titanic_df3[col] = le.fit_transform(titanic_df3[col])

# 2. OneHotEncoder 적용할 컬럼
ohe_cols = ['sex', 'embarked', 'who', 'deck', 'embark_town']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 결과를 저장할 DataFrame
ohe_result = pd.DataFrame(index=titanic_df3.index)

for col in ohe_cols:
    #encoded 타입은 ndarray
    encoded = ohe.fit_transform(titanic_df3[[col]])
    col_names = ohe.get_feature_names_out([col])
    encoded_df = pd.DataFrame(encoded, columns=col_names, index=titanic_df3.index)
    ohe_result = pd.concat([ohe_result, encoded_df], axis=1)

# 기존 원본 컬럼 삭제 후 병합
titanic_df3 = pd.concat([titanic_df3.drop(columns=ohe_cols), ohe_result], axis=1)
print(titanic_df3.info())


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   survived                 891 non-null    int64  
 1   pclass                   891 non-null    int64  
 2   age                      891 non-null    float64
 3   sibsp                    891 non-null    int64  
 4   parch                    891 non-null    int64  
 5   fare                     891 non-null    float64
 6   class                    891 non-null    int64  
 7   adult_male               891 non-null    int64  
 8   alive                    891 non-null    str    
 9   alone                    891 non-null    int64  
 10  sex_female               891 non-null    float64
 11  sex_male                 891 non-null    float64
 12  embarked_C               891 non-null    float64
 13  embarked_Q               891 non-null    float64
 14  embarked_S               891 non-null

데이터 전처리를 단계별로 진행할 때 범주형 변수와 연속형 변수를 나누고, 개별적으로 전처리하는 과정을 했다.
sklean-learn에서 제공하는 make_column_transformer와 ColumnTransformer을 이용해서 범주형변수와 연속형 변수를 한번에  전처리를 할 수 있도록 제공해준다.

make_column_transformer를 이용한 인코딩
- make_column_transformer((전처리 객체, 전처리를 적용할 컬럼), (전처리 객체, 전처리를 적용할 컬럼))
- LabelEncoder는 1차원만 처리 가능하므로 make_column_transformer에 직접 쓸 수 없습니다. 대신 **OrdinalEncoder**를 사용한다.

### make_column_transformer

In [115]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

titanic_df4 = titanic.copy()

In [116]:
titanic_df4.drop(columns=['alive'], inplace=True)

In [ ]:
titanic_df4.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alone'],
      dtype='str')

In [117]:
onehot_cols = ['sex', 'embarked', 'who', 'deck', 'embark_town']
ordinal_cols = ['class', 'adult_male', 'alone']

# make_column_transformer()를 이용하여 전처리기 생성
# remainder='passthrough' 옵션을 지정하면, 지정한 컬럼 외의 나머지 컬럼은 그대로 유지
column_transformer = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown='ignore'), onehot_cols),
    (OrdinalEncoder(), ordinal_cols),
    remainder='passthrough'
)


# 변환적용
encoded = column_transformer.fit_transform(titanic_df4)

In [118]:
# encoded
# titanic_df4

# 컬럼 이름 추출
print(dir(column_transformer))
print(column_transformer.named_transformers_)
print(dir(column_transformer.named_transformers_['onehotencoder']))


onehot_features_names = column_transformer.named_transformers_['onehotencoder'].get_feature_names_out(onehot_cols)
print(f'onehot_feature_names:{onehot_features_names}')

ordinal_features_names = ordinal_cols 
print(f'ordinal_feature_names:{ordinal_features_names}')

other_features_names = [col for col in titanic_df4.columns if col not in onehot_cols + ordinal_cols]
print(f'other_features_names:{other_features_names}')

print(type(onehot_features_names)) # class 'numpy.ndarray'
print(type(ordinal_features_names)) # class 'list'
print(type(other_features_names)) # class 'list'

final_features_names = list(onehot_features_names) + ordinal_features_names + other_features_names
print(f'final_features_names:{final_features_names}')

# # DataFrame으로 변환
titanic_df4 = pd.DataFrame(encoded, columns=final_features_names, index=titanic_df4.index)
print(titanic_df4.info())


['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_prefix_for_feature_names_out', '_call_func_on_transformers', '_check_estimators_are_instances', '_columns', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_get_class_level_metadata_request_values', '_get_doc_link', '_get_empty_routing', '_get_feature_name_out_for_transformer', '_get_fitted_attr_html', '_get_metadata_request', '_get_param_names', '_get_params', '_get_params_html', '_get_remainder_cols', '_get_remainder_cols_dtype', '_hstack', '_html_repr', '_iter', '_log_me

In [119]:
feature_names = column_transformer.get_feature_names_out()
print(f'feature_names:{feature_names}')

feature_names:['onehotencoder__sex_female' 'onehotencoder__sex_male'
 'onehotencoder__embarked_C' 'onehotencoder__embarked_Q'
 'onehotencoder__embarked_S' 'onehotencoder__who_child'
 'onehotencoder__who_man' 'onehotencoder__who_woman'
 'onehotencoder__deck_A' 'onehotencoder__deck_B' 'onehotencoder__deck_C'
 'onehotencoder__deck_D' 'onehotencoder__deck_E' 'onehotencoder__deck_F'
 'onehotencoder__deck_G' 'onehotencoder__deck_nan'
 'onehotencoder__embark_town_Cherbourg'
 'onehotencoder__embark_town_Queenstown'
 'onehotencoder__embark_town_Southampton' 'onehotencoder__embark_town_nan'
 'ordinalencoder__class' 'ordinalencoder__adult_male'
 'ordinalencoder__alone' 'remainder__survived' 'remainder__pclass'
 'remainder__age' 'remainder__sibsp' 'remainder__parch' 'remainder__fare']


In [120]:
titanic_df5 = titanic.copy()
column_transformer_ignore = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
    (OrdinalEncoder(), ordinal_cols),
    remainder='passthrough',
    verbose_feature_names_out=False  # 접두사 제거 옵션
)

encoded = column_transformer_ignore.fit_transform(titanic_df5)
feature_names = column_transformer_ignore.get_feature_names_out()

encoded_df = pd.DataFrame(encoded, columns=feature_names)
print(f'encoded_df :{encoded_df.columns}')
print(encoded_df)

encoded_df :Index(['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S',
       'who_child', 'who_man', 'who_woman', 'deck_A', 'deck_B', 'deck_C',
       'deck_D', 'deck_E', 'deck_F', 'deck_G', 'deck_nan',
       'embark_town_Cherbourg', 'embark_town_Queenstown',
       'embark_town_Southampton', 'embark_town_nan', 'class', 'adult_male',
       'alone', 'survived', 'pclass', 'age', 'sibsp', 'parch', 'fare',
       'alive'],
      dtype='str')
    sex_female sex_male embarked_C embarked_Q embarked_S who_child who_man  \
0          0.0      1.0        0.0        0.0        1.0       0.0     1.0   
1          1.0      0.0        1.0        0.0        0.0       0.0     0.0   
2          1.0      0.0        0.0        0.0        1.0       0.0     0.0   
3          1.0      0.0        0.0        0.0        1.0       0.0     0.0   
4          0.0      1.0        0.0        0.0        1.0       0.0     1.0   
..         ...      ...        ...        ...        ...       ...     